In [ ]:
# ---------------------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [ ]:
# -----------------------------------------------------------------------------
# STEP 1.0 - Creating 4 different DGPs
# -----------------------------------------------------------------------------

# Seed -> same random numbers every run -> reproducible results
SEED = 1024
rng = np.random.default_rng(SEED)

# Sample sizes (slide: N_T, N_P >= 100'000)
N_T = 100_000   # training sample T
N_P = 100_000   # prediction / validation sample P

# True parameters (fixed in all DGPs)
ALPHA0 = 1.0    # true causal effect of D on Y  -> what we want to recover
BETA0 = 1.0     # scale of X

# The 4 DGPs: lambda0 in {0, 5}, gamma0 in {1, 2}
DGPS = {
    1: {"lam": 0, "gamma": 1},   
    2: {"lam": 0, "gamma": 2},   
    3: {"lam": 5, "gamma": 1},   
    4: {"lam": 5, "gamma": 2},   
}


# -----------------------------------------------------------------------------
# STEP 1.1 - Generate Data Sets for each DGPs
# -----------------------------------------------------------------------------
def generate(N, lam, gamma, rng):
    """
    y_i = 1 + d_i*alpha0 + (x_i*beta0)^gamma0 + u_i
    d_i = 1(x_i*lambda0 + v_i > 0)
    u, v, x ~ N(0,1) iid
    """
    x = rng.standard_normal(N)   # baseline health index
    v = rng.standard_normal(N)   # random part of the treatment decision
    u = rng.standard_normal(N)   # unobserved factors affecting the outcome

    d = (x * lam + v > 0).astype(int)                 # 1 = took the drug
    y = 1 + d * ALPHA0 + (x * BETA0) ** gamma + u     # health score after treatment

    # u and v are NOT stored: the researcher never observes them
    return pd.DataFrame({"y": y, "d": d, "x": x})


# Generate T and P for each DGP -> data[k]["T"], data[k]["P"]
data = {}
for k, p in DGPS.items():
    data[k] = {
        "T": generate(N_T, p["lam"], p["gamma"], rng),
        "P": generate(N_P, p["lam"], p["gamma"], rng),
    }

In [ ]:
# -----------------------------------------------------------------------------
# STEP 2 - Prediction with OLS (train on T, predict on T and P, RMSE)
# -----------------------------------------------------------------------------

# The 2 specifications = which columns go into the regression
SPECS = {
    "A": ["d", "x"],          # Y = b0 + b1*D + b2*X
    "B": ["d", "x", "x2"],    # Y = b0 + b1*D + b2*X + b3*X^2
}

def design(df, cols):
    """Build the regressor matrix: constant + chosen columns."""
    df = df.copy()
    df["x2"] = df["x"] ** 2
    return sm.add_constant(df[cols])

def rmse(y, y_hat):
    return np.sqrt(np.mean((y - y_hat) ** 2))

results = []
for k in DGPS:
    T, P = data[k]["T"], data[k]["P"]
    for s, cols in SPECS.items():
        # 1) Train OLS on T only
        X_T = design(T, cols)
        model = sm.OLS(T["y"], X_T).fit()

        # 2) Predict Y on T and on P (with coefficients from T)
        y_hat_T = model.predict(X_T)
        y_hat_P = model.predict(design(P, cols))

        # 3) RMSE on T and P
        results.append({
            "DGP": k,
            "lambda0": DGPS[k]["lam"],
            "gamma0": DGPS[k]["gamma"],
            "spec": s,
            "RMSE_T": rmse(T["y"], y_hat_T),
            "RMSE_P": rmse(P["y"], y_hat_P),
        })

results = pd.DataFrame(results)
print(results.round(4).to_string(index=False))
# When the dgp is liner (γ0=1) spec A is better, using spec B is ok. 
# When the dgp is quadratic (γ0=2) spec B is better, using spec A is not great as x^2 is in the true dgp but is missing from the model.

 DGP  lambda0  gamma0 spec  RMSE_T  RMSE_P
   1        0       1    A  1.0038  1.0014
   1        0       1    B  1.0037  1.0014
   2        0       2    A  1.7328  1.7456
   2        0       2    B  0.9994  0.9984
   3        5       1    A  1.0021  1.0001
   3        5       1    B  1.0021  1.0001
   4        5       2    A  1.7251  1.7133
   4        5       2    B  1.0024  1.0005


In [ ]:
# -----------------------------------------------------------------------------
# STEP 3 - Causal effect of D on Y (coefficient on d vs true ALPHA0)
# -----------------------------------------------------------------------------
causal = []
for k in DGPS:
    T = data[k]["T"]
    for s, cols in SPECS.items():
        model = sm.OLS(T["y"], design(T, cols)).fit()

        alpha_hat = model.params["d"]    # estimated causal effect
        se = model.bse["d"]              # standard error of alpha_hat

        causal.append({
            "DGP": k,
            "lambda0": DGPS[k]["lam"],
            "gamma0": DGPS[k]["gamma"],
            "spec": s,
            "alpha_hat": alpha_hat,
            "diff (alpha_hat - alpha0)": alpha_hat - ALPHA0,
            "se": se,
            "t (diff/se)": (alpha_hat - ALPHA0) / se,
        })

causal = pd.DataFrame(causal)
print(causal.round(4).to_string(index=False))
# When the dgp is quadratic (γ0=2) spec A is not great as x^2 goes into the error term
# Precision lowers significantly when λ0 = 5 (dgp 3 and 4).

 DGP  lambda0  gamma0 spec  alpha_hat  diff (alpha_hat - alpha0)     se  t (diff/se)
   1        0       1    A     0.9932                    -0.0068 0.0063      -1.0636
   1        0       1    B     0.9933                    -0.0067 0.0063      -1.0596
   2        0       2    A     0.9992                    -0.0008 0.0110      -0.0743
   2        0       2    B     0.9939                    -0.0061 0.0063      -0.9662
   3        5       1    A     1.0053                     0.0053 0.0102       0.5145
   3        5       1    B     1.0053                     0.0053 0.0102       0.5160
   4        5       2    A     1.0055                     0.0055 0.0175       0.3152
   4        5       2    B     1.0107                     0.0107 0.0102       1.0489
